### Libraries

In [ ]:
library(Seurat)
library(DESeq2)
library(readr)
library(dplyr)
library(tibble)
library(readxl)
library(pheatmap)
library(writexl)
library(ggplot2)
library(RColorBrewer)
library(tximport)
library(GenomicFeatures)
library(stringr)
library(openxlsx)
library(ape)
library(msigdbr)
library(fgsea)
library(enrichplot)
library(DOSE)
library(clusterProfiler)
library(stringr)
library(vegan)
library(tidyr)
library(tools)
library(ggpubr)  # For stat_cor
library(sva)

### NES Heatmaps

In [ ]:
vtr_df = read_excel('/rprojectnb/cancergrp/brb/raw_data/VTR_INFORMATION.xlsx')
vtr_codebook = read_excel('/rprojectnb/cancergrp/brb/raw_data/VTR_codebook.xlsx', sheet = 1)


folder_path1 <- paste0("/rprojectnb/cancergrp/brb/reproducibility/intermediate_files/GSEA_objects/")
folder_path <- paste0("/rprojectnb/cancergrp/brb/reproducibility/intermediate_files/GSEA_objects/heatmaps/NES/samples_vs_samples/")
table_file <- paste0(folder_path1, "master_table.xlsx")
pvalue_file <- paste0(folder_path1, "pvalue_table.xlsx")

if (!dir.exists(folder_path)) {
    dir.create(folder_path, recursive = TRUE)
}

for (condition_id in c("stimulated", "unstimulated")) {

    score_matrix_saved <- read.xlsx(table_file)
    p_value_matrix <- read.xlsx(pvalue_file)
             
    # Make sure the column names of both matrices match before proceeding (excluding identifier columns)
    if (!all(colnames(score_matrix_saved)[-1] == colnames(p_value_matrix)[-1])) {
        stop("The column names of the score and p-value matrices do not match.")
    }
    
    # Exclude the "Paths" column from the numeric operations and retain it separately
    score_paths <- score_matrix_saved[,"Paths"]
    numeric_score_matrix <- score_matrix_saved[,-which(colnames(score_matrix_saved) == "Paths")]
    numeric_pvalue_matrix <- p_value_matrix[,-which(colnames(p_value_matrix) == "Paths")]

    numeric_score_matrix_w_0 <- numeric_score_matrix

    numeric_score_matrix_no_0 <- numeric_score_matrix
    
    # Set NES values to 0 where p-value is greater than 0.1
    numeric_score_matrix_w_0[numeric_pvalue_matrix > 0.05] <- 0
    
    # Reattach the "Paths" column to the modified numeric matrix
    score_matrix_no_0 <- cbind(Paths = score_paths, numeric_score_matrix_no_0)
    score_matrix_w_0 <- cbind(Paths = score_paths, numeric_score_matrix_w_0)
    
    # Filter columns by condition (if necessary)
    score_matrix_no_0 <- score_matrix_no_0[, c("Paths", grep(paste0("_", condition_id), 
                                                               colnames(score_matrix_no_0), 
                                                               value = TRUE))]
    score_matrix_w_0 <- score_matrix_w_0[, c("Paths", grep(paste0("_", condition_id), 
                                                               colnames(score_matrix_w_0), 
                                                               value = TRUE))]
        
    # Exclude the first column if it’s an identifier and keep only numeric columns
    data_matrix_no_0 <- as.matrix(score_matrix_no_0[,-1])
    data_matrix_w_0 <- as.matrix(score_matrix_w_0[,-1])

    matrices_0 <- list(with_0 = data_matrix_w_0, no_0 = data_matrix_no_0)
    
    # Create empty Pearson correlation allData
    empty_matrix <- matrix(NA, ncol = ncol(data_matrix_no_0), nrow = ncol(data_matrix_no_0))
    colnames(empty_matrix) <- colnames(data_matrix_no_0)
    rownames(empty_matrix) <- colnames(data_matrix_no_0)

    cor_matrix_w_0 <- empty_matrix
    cor_matrix_no_0 <- empty_matrix
    concord_matrix_w_0 <- empty_matrix
    concord_matrix_no_0 <- empty_matrix

    result_matrices <- list(correlation = list(with_0 = cor_matrix_w_0, no_0 = cor_matrix_no_0),
                           concordance = list(with_0 = concord_matrix_w_0, no_0 = concord_matrix_no_0))
    
    # Calculate pairwise correlations
    all_combinations <- combn(colnames(data_matrix_no_0), 2, simplify = FALSE)
    for (pair in all_combinations) {

        for (matrix_i in 1:length(matrices_0)) {

            x_col <- pair[1]
            y_col <- pair[2]
            
            # Extract the column values
            x_values <- matrices_0[[matrix_i]][, x_col]
            y_values <- matrices_0[[matrix_i]][, y_col]

            ns_values_x <- rep(FALSE, length(score_paths))
            ns_values_y <- rep(FALSE, length(score_paths))

            # Match score_path with score_matrix Paths to make sure they're aligned
            ns_values_x[match(score_paths, score_matrix_no_0$Paths)[which(numeric_pvalue_matrix[, x_col] > 0.05)]] <- TRUE
            ns_values_y[match(score_paths, score_matrix_no_0$Paths)[which(numeric_pvalue_matrix[, y_col] > 0.05)]] <- TRUE

            if (matrix_i == 1) {
                # Set NA where both x and y are 0 (to ignore these pairs)
                x_values[x_values == 0 & y_values == 0] <- NA
                y_values[x_values == 0 & y_values == 0] <- NA

            } else {
                # Set NA where both x and y are non significant (to ignore these pairs)
                x_values[ns_values_x & ns_values_y] <- NA
                y_values[ns_values_x & ns_values_y] <- NA
                
            }

            ##### CONCORDANCE ####
            
            # Determine C and D values
            C <- sum((x_values > 0 & y_values > 0) | (x_values < 0 & y_values < 0), na.rm = TRUE) # Same sign count
            D <- sum((x_values > 0 & y_values < 0) | (x_values < 0 & y_values > 0) |
                     (x_values == 0 & y_values != 0) | (x_values != 0 & y_values == 0), na.rm = TRUE) # Different sign or zero pairs (excluding both-zero pairs)
            
            score_all <- (C - D) / (C + D)
            
            #### PEARSON CORRELATION ####

            # Create a logical index for non-NA and non-zero values in x and y
            non_na_idx <- !is.na(x_values) & !is.na(y_values)
            
            # Calculate correlation for all pairs (excluding rows with NA values)
            correlation_all <- cor(x_values[non_na_idx], y_values[non_na_idx], use = "pairwise.complete.obs")
            
            # Store the correlation/concordance coefficient in the matrix (symmetric)

            result_matrices$correlation[[matrix_i]][x_col, y_col] <- correlation_all
            result_matrices$concordance[[matrix_i]][x_col, y_col] <- score_all

            result_matrices$correlation[[matrix_i]][y_col, x_col] <- correlation_all
            result_matrices$concordance[[matrix_i]][y_col, x_col] <- score_all

            diag(result_matrices$concordance[[matrix_i]]) <- 1
            diag(result_matrices$correlation[[matrix_i]]) <- 1

        }
        
    }

    for (heatmap_i in 1:length(result_matrices)) {

        for (matrix_i in 1:length(result_matrices[[heatmap_i]])) {
    
            metadata_col <- data.frame(id = colnames(result_matrices[[heatmap_i]][[matrix_i]]))
            metadata_col$batch_id = str_split_fixed(metadata_col$id, '\\_', 4)[,3]
            metadata_col$condition = str_split_fixed(metadata_col$id, '\\_', 4)[,2]
            metadata_col$vtr = str_split_fixed(metadata_col$id, '\\_', 4)[,1]
            metadata_col$virus_type <- ifelse(
                                              metadata_col$vtr == "ctrl", 
                                              "ctrl", 
                                              vtr_df$`Viral family`[match(metadata_col$vtr, vtr_df$code)]
                                            )
            rownames(metadata_col) = metadata_col$id
            
            # Map `Virus` from `vtr_df` based on `vtr` code
            metadata_col$virus_name <- ifelse(
                metadata_col$vtr == "ctrl", 
                "ctrl", 
                vtr_codebook$`Virus`[match(metadata_col$vtr, vtr_codebook$code)]
            )
              
            # Create the new combined labels (e.g., "VTR10 (Virus)")
            metadata_col$combined_label <- ifelse(
                metadata_col$vtr == "ctrl", 
                "ctrl", 
                paste0(metadata_col$vtr, " (", metadata_col$virus_name, ")")
            )
            
            # Assign the combined labels to row and column names for the heatmap
              # row_labels <- metadata_col$combined_label
              # col_labels <- metadata_col$combined_label
            
            # Crete palette colors
            n_virus_type = length(unique(metadata_col$virus_type))
            n_batch_id = length(unique(metadata_col$batch_id))
            n_condition = length(unique(metadata_col$condition))
            
            ann_colors <- list(
                virus_type = setNames(colorRampPalette(brewer.pal(n_virus_type, "Set3"))(n_virus_type), unique(metadata_col$virus_type)),
                batch_id = setNames(colorRampPalette(brewer.pal(n_batch_id, "Dark2"))(n_batch_id), unique(metadata_col$batch_id)),
                condition = setNames(colorRampPalette(brewer.pal(n_condition, "Paired"))(n_condition), unique(metadata_col$condition))
            )
            
            my_colors <- colorRampPalette(c("navy", "white", "firebrick3"))(100)

            if (heatmap_i == 1) {
                if (matrix_i == 1) {
                    heatmap_folder <- paste0(folder_path, "correlation")
                    heatmap_file <- paste0(heatmap_folder, "/nes_correlation_heatmap_w_0_")
                    heatmap_title <- "Correlation Heatmap"
                } else {
                    heatmap_folder <- paste0(folder_path, "correlation")
                    heatmap_title <- "Correlation Heatmap"
                    heatmap_file <- paste0(heatmap_folder, "/nes_correlation_heatmap_no_0_")
                }
            } else if (heatmap_i == 2) {
                if (matrix_i == 1) {
                    heatmap_folder <- paste0(folder_path, "concordance")
                    heatmap_title <- "Concordance Heatmap"
                    heatmap_file <- paste0(heatmap_folder, "/nes_concordance_heatmap_w_0_")
                } else {
                    heatmap_folder <- paste0(folder_path, "concordance")
                    heatmap_title <- "Concordance Heatmap"
                    heatmap_file <- paste0(heatmap_folder, "/nes_concordance_heatmap_no_0_")
                }
            }

            heatmap_file <- paste0(heatmap_file, condition_id)

            if (!dir.exists(heatmap_folder)) {
              dir.create(heatmap_folder, recursive = TRUE)
            }
            
            ## Specify breaks
            rg <- max(abs(result_matrices[[heatmap_i]][[matrix_i]]))

            if (!is.finite(rg)) {
                rg <- 1  # Set rg to a fixed value (can use 1 for the upper limit)
            }

            vtr_annotations <- list(c('virus_type', 'batch_id'), c('virus_type'))
            
            for (heatmap_j in 1:length(vtr_annotations)) {
                
                ## Plot heatmap
                out <- pheatmap(result_matrices[[heatmap_i]][[matrix_i]],
                         clustering_distance_rows = "euclidean",
                         cluster_rows = TRUE,
                         # cluster_cols = FALSE,
                         breaks = seq(-rg, rg, length.out = 100),
                         clustering_distance_cols = "euclidean",
                         clustering_method = "complete",
                         # labels_row = row_labels,
                         # labels_col = col_labels,
                         color = my_colors,
                         show_rownames = FALSE,
                         show_colnames = FALSE,
                         annotation_col = metadata_col[, vtr_annotations[[heatmap_j]], drop = FALSE],
                         annotation_colors = ann_colors,
                         border_color  = 'black',
                         main = heatmap_title,
                         fontsize = 8,
                         filename = paste0(heatmap_file, "_", heatmap_j, ".pdf"),
                         width = 12,
                         height = 10)
            }

            reordered_matrix <- result_matrices[[heatmap_i]][[matrix_i]][out$tree_row[["order"]], out$tree_col[["order"]]]
            
            write_xlsx(data.frame(VTR = rownames(reordered_matrix), reordered_matrix, check.names = FALSE), paste0(heatmap_file, ".xlsx"))
        }
    }
}

### L2FC Heatmaps

In [ ]:
xlsx_files <- list.files(path = "/rprojectnb/cancergrp/brb/reproducibility/intermediate_files/DGE_objects/filtered_tables/", pattern = 'strategy2', full.names = TRUE)

In [ ]:
# Initialize an empty data frame to store log2FoldChange values
combined_df <- data.frame()
pvalue_df <- data.frame()

# Loop through each file and extract log2FoldChange and pvalue columns
for (file in xlsx_files) {
    # Read the file into a data frame
    df <- read_xlsx(file)
    
    # Check if the required columns exist in the file (e.g., symbol, log2FoldChange, and pvalue)
    if ("log2FoldChange" %in% colnames(df) & "symbol" %in% colnames(df) & "pvalue" %in% colnames(df)) {
        
        # Extract the base name of the file to use as a prefix for the columns
        file_name <- file_path_sans_ext(basename(file))
        
        # Extract and rename log2FoldChange column
        temp_log2FC_df <- df %>%
            select(symbol, log2FoldChange) %>%
            rename(!!paste0(file_name, "_log2FC") := log2FoldChange)
        
        # Extract and rename pvalue column
        temp_pvalue_df <- df %>%
            select(symbol, pvalue) %>%
            rename(!!paste0(file_name, "_pvalue") := pvalue)
        
        # Merge log2FoldChange values with combined_df by 'symbol'
        if (nrow(combined_df) == 0) {
            combined_df <- temp_log2FC_df
        } else {
            combined_df <- full_join(combined_df, temp_log2FC_df, by = "symbol")
        }

        # Merge pvalue values with pvalue_df by 'symbol'
        if (nrow(pvalue_df) == 0) {
            pvalue_df <- temp_pvalue_df
        } else {
            pvalue_df <- full_join(pvalue_df, temp_pvalue_df, by = "symbol")
        }
    }
}

print(dim(combined_df))

In [ ]:
folder_path <- paste0("/rprojectnb/cancergrp/brb/reproducibility/intermediate_files/DGE_objects/heatmaps/L2FC/samples_vs_samples/")
vtr_df = read_excel('/rprojectnb/cancergrp/brb/raw_data/VTR_INFORMATION.xlsx')
vtr_codebook = read_excel('/rprojectnb/cancergrp/brb/raw_data/VTR_codebook.xlsx', sheet = 1)
xlsx_files <- list.files(path = "/rprojectnb/cancergrp/brb/reproducibility/intermediate_files/DGE_objects/filtered_tables/", pattern = 'strategy2', full.names = TRUE)

for (condition_id in c("stimulated", "unstimulated")) {

    score_matrix_saved <- combined_df
    p_value_matrix <- pvalue_df

    # Make sure the column names of both matrices match before proceeding (excluding identifier columns)
    # Extract the shared part of the column names (everything before the last underscore)
    score_cols <- sub("_log2FC$", "", colnames(score_matrix_saved)[-1])
    pvalue_cols <- sub("_pvalue$", "", colnames(p_value_matrix)[-1])
    
    # Check if the relevant parts of the column names match
    if (!all(score_cols == pvalue_cols)) {
        stop("The column names of the score and p-value matrices do not match.")
    }
    
    # Exclude the "Paths" column from the numeric operations and retain it separately
    score_paths <- score_matrix_saved[,"symbol"]
    numeric_score_matrix <- score_matrix_saved[,-which(colnames(score_matrix_saved) == "symbol")]
    numeric_pvalue_matrix <- p_value_matrix[,-which(colnames(p_value_matrix) == "symbol")]

    numeric_score_matrix_w_0 <- numeric_score_matrix

    numeric_score_matrix_no_0 <- numeric_score_matrix
    
    # Set L2FC values to 0 where p-value is greater than 0.1
    numeric_score_matrix_w_0[numeric_pvalue_matrix > 0.05] <- 0
    
    # Reattach the "Paths" column to the modified numeric matrix
    score_matrix_no_0 <- cbind(Paths = score_paths, numeric_score_matrix_no_0)
    score_matrix_w_0 <- cbind(Paths = score_paths, numeric_score_matrix_w_0)
    
    # Filter columns by condition (if necessary)
    score_matrix_no_0 <- score_matrix_no_0[, c("symbol", grep(paste0("_", condition_id), 
                                                               colnames(score_matrix_no_0), 
                                                               value = TRUE))]
    score_matrix_w_0 <- score_matrix_w_0[, c("symbol", grep(paste0("_", condition_id), 
                                                               colnames(score_matrix_w_0), 
                                                               value = TRUE))]
    
    # Exclude the first column if it’s an identifier and keep only numeric columns
    data_matrix_no_0 <- as.matrix(score_matrix_no_0[,-1])
    data_matrix_w_0 <- as.matrix(score_matrix_w_0[,-1])

    matrices_0 <- list(with_0 = data_matrix_w_0, no_0 = data_matrix_no_0)
    
    # Create empty Pearson correlation allData
    empty_matrix <- matrix(NA, ncol = ncol(data_matrix_no_0), nrow = ncol(data_matrix_no_0))
    colnames(empty_matrix) <- colnames(data_matrix_no_0)
    rownames(empty_matrix) <- colnames(data_matrix_no_0)

    cor_matrix_w_0 <- empty_matrix
    cor_matrix_no_0 <- empty_matrix
    concord_matrix_w_0 <- empty_matrix
    concord_matrix_no_0 <- empty_matrix

    result_matrices <- list(correlation = list(with_0 = cor_matrix_w_0, no_0 = cor_matrix_no_0),
                           concordance = list(with_0 = concord_matrix_w_0, no_0 = concord_matrix_no_0))
    
    # Calculate pairwise correlations
    all_combinations <- combn(colnames(data_matrix_no_0), 2, simplify = FALSE)
    for (pair in all_combinations) {

        for (matrix_i in 1:length(matrices_0)) {

            x_col <- pair[1]
            y_col <- pair[2]
            
            # Extract the column values
            x_values <- matrices_0[[matrix_i]][, x_col]
            y_values <- matrices_0[[matrix_i]][, y_col]

            ns_values_x <- rep(FALSE, length(score_paths))
            ns_values_y <- rep(FALSE, length(score_paths))

            # Match score_path with score_matrix Paths to make sure they're aligned
            ns_values_x[match(score_paths, score_matrix_no_0$symbol)
                        [which(numeric_pvalue_matrix[, gsub('log2FC', 'pvalue', x_col)] > 0.05)]] <- TRUE
            ns_values_y[match(score_paths, score_matrix_no_0$symbol)
                        [which(numeric_pvalue_matrix[, gsub('log2FC', 'pvalue', y_col)] > 0.05)]] <- TRUE
            

            if (matrix_i == 1) {
                # Set NA where both x and y are 0 (to ignore these pairs)
                x_values[x_values == 0 & y_values == 0] <- NA
                y_values[x_values == 0 & y_values == 0] <- NA

            } else {
                # Set NA where both x and y are 0 (to ignore these pairs)
                x_values[ns_values_x & ns_values_y] <- NA
                y_values[ns_values_x & ns_values_y] <- NA

            }

            ##### CONCORDANCE ####
            
            # Determine C and D values
            C <- sum((x_values > 0 & y_values > 0) | (x_values < 0 & y_values < 0), na.rm = TRUE) # Same sign count
            D <- sum((x_values > 0 & y_values < 0) | (x_values < 0 & y_values > 0) |
                     (x_values == 0 & y_values != 0) | (x_values != 0 & y_values == 0), na.rm = TRUE) # Different sign or zero pairs (excluding both-zero pairs)
            
            score_all <- (C - D) / (C + D)
            
            #### PEARSON CORRELATION ####

            # Create a logical index for non-NA and non-zero values in x and y
            non_na_idx <- !is.na(x_values) & !is.na(y_values)
            
            # Calculate correlation for all pairs (excluding rows with NA values)
            correlation_all <- cor(x_values[non_na_idx], y_values[non_na_idx], use = "pairwise.complete.obs")
            
            # Store the correlation/concordance coefficient in the matrix (symmetric)

            result_matrices$correlation[[matrix_i]][x_col, y_col] <- correlation_all
            result_matrices$concordance[[matrix_i]][x_col, y_col] <- score_all

            result_matrices$correlation[[matrix_i]][y_col, x_col] <- correlation_all
            result_matrices$concordance[[matrix_i]][y_col, x_col] <- score_all

            diag(result_matrices$concordance[[matrix_i]]) <- 1
            diag(result_matrices$correlation[[matrix_i]]) <- 1

        }
        
    }

    for (heatmap_i in 1:length(result_matrices)) {

        for (matrix_i in 1:length(result_matrices[[heatmap_i]])) {
    
            metadata_col <- data.frame(id = colnames(result_matrices[[heatmap_i]][[matrix_i]]))
            metadata_col$batch_id = str_split_fixed(metadata_col$id, '\\_', 4)[,3]
            metadata_col$condition = str_split_fixed(metadata_col$id, '\\_', 4)[,2]
            metadata_col$vtr = str_split_fixed(metadata_col$id, '\\_', 4)[,1]
            # metadata_col$virus_type = vtr_df$`Viral family`[match(metadata_col$vtr, vtr_df$code)]
            metadata_col$virus_type <- ifelse(
                                              metadata_col$vtr == "ctrl", 
                                              "ctrl", 
                                              vtr_df$`Viral family`[match(metadata_col$vtr, vtr_df$code)]
                                            )
            rownames(metadata_col) = metadata_col$id
            # Map `Virus` from `vtr_df` based on `vtr` code
             metadata_col$virus_name <- ifelse(
                metadata_col$vtr == "ctrl", 
                "ctrl", 
                vtr_codebook$`Virus`[match(metadata_col$vtr, vtr_codebook$code)]
              )
              
              # Create the new combined labels (e.g., "VTR10 (Virus)")
              metadata_col$combined_label <- ifelse(
                metadata_col$vtr == "ctrl", 
                "ctrl", 
                paste0(metadata_col$vtr, " (", metadata_col$virus_name, ")")
              )
            # Assign the combined labels to row and column names for the heatmap
              # row_labels <- metadata_col$combined_label
              # col_labels <- metadata_col$combined_label
            
            # Crete palette colors
            n_virus_type = length(unique(metadata_col$virus_type))
            n_batch_id = length(unique(metadata_col$batch_id))
            n_condition = length(unique(metadata_col$condition))
            
            ann_colors <- list(
                virus_type = setNames(colorRampPalette(brewer.pal(n_virus_type, "Set3"))(n_virus_type), unique(metadata_col$virus_type)),
                batch_id = setNames(colorRampPalette(brewer.pal(n_batch_id, "Dark2"))(n_batch_id), unique(metadata_col$batch_id)),
                condition = setNames(colorRampPalette(brewer.pal(n_condition, "Paired"))(n_condition), unique(metadata_col$condition))
            )
            my_colors <- colorRampPalette(c("navy", "white", "firebrick3"))(100)

            if (heatmap_i == 1) {
                if (matrix_i == 1) {
                    heatmap_folder <- paste0(folder_path, "correlation")
                    heatmap_file <- paste0(heatmap_folder, "/l2fc_correlation_heatmap_w_0_")
                    heatmap_title <- "Correlation Heatmap"
                } else {
                    heatmap_folder <- paste0(folder_path, "correlation")
                    heatmap_title <- "Correlation Heatmap"
                    heatmap_file <- paste0(heatmap_folder, "/l2fc_correlation_heatmap_no_0_")
                }
            } else if (heatmap_i == 2) {
                if (matrix_i == 1) {
                    heatmap_folder <- paste0(folder_path, "concordance")
                    heatmap_title <- "Concordance Heatmap"
                    heatmap_file <- paste0(heatmap_folder, "/l2fc_concordance_heatmap_w_0_")
                } else {
                    heatmap_folder <- paste0(folder_path, "concordance")
                    heatmap_title <- "Concordance Heatmap"
                    heatmap_file <- paste0(heatmap_folder, "/l2fc_concordance_heatmap_no_0_")
                }
            }

            heatmap_file <- paste0(heatmap_file, condition_id)

            if (!dir.exists(heatmap_folder)) {
              dir.create(heatmap_folder, recursive = TRUE)
            }
            
            ## Specify breaks
            rg <- max(abs(result_matrices[[heatmap_i]][[matrix_i]]))

            if (!is.finite(rg)) {
                rg <- 1  # Set rg to a fixed value (can use 1 for the upper limit)
            }

            vtr_annotations <- list(c('virus_type', 'batch_id'), c('virus_type'))
            
            for (heatmap_j in 1:length(vtr_annotations)) {
                
                ## Plot heatmap
                out <- pheatmap(result_matrices[[heatmap_i]][[matrix_i]],
                         clustering_distance_rows = "euclidean",
                         cluster_rows = TRUE,
                         # cluster_cols = FALSE,
                         breaks = seq(-rg, rg, length.out = 100),
                         clustering_distance_cols = "euclidean",
                         clustering_method = "complete",
                         # labels_row = row_labels,
                         # labels_col = col_labels,
                         color = my_colors,
                         show_rownames = FALSE,
                         show_colnames = FALSE,
                         annotation_col = metadata_col[, vtr_annotations[[heatmap_j]], drop = FALSE],
                         annotation_colors = ann_colors,
                         border_color  = 'black',
                         main = heatmap_title,
                         fontsize = 8,
                         filename = paste0(heatmap_file, "_", heatmap_j, ".pdf"),
                         width = 12,
                         height = 10)
            }

            reordered_matrix <- result_matrices[[heatmap_i]][[matrix_i]][out$tree_row[["order"]], out$tree_col[["order"]]]
            
            write_xlsx(data.frame(VTR = rownames(reordered_matrix), reordered_matrix, check.names = FALSE), paste0(heatmap_file, ".xlsx"))
        }
    }
}

In [ ]:
msigdb <- msigdbr(species = 'Homo sapiens', category = 'C2')
msigdb <- msigdb %>% filter(gs_subcat %in% c('CP:REACTOME'))
unique(msigdb$gs_subcat)

In [ ]:
# Step 1: Create a list of gene sets for each pathway
pathway_gene_sets <- msigdb %>%
  group_by(gs_name) %>%
  summarise(genes = list(unique(gene_symbol))) %>%
  deframe()

In [ ]:
# Step 2: Define Similarity function
similarity_by_min <- function(set1, set2) {
  intersection <- length(intersect(set1, set2))
  min_size <- min(length(set1), length(set2))
  
  if (min_size == 0) {
    return(0)  # Avoid division by zero
  } else {
    return(intersection / min_size)
  }
}

In [ ]:
# Step 3: Calculate pairwise Jaccard similarities
pathway_names <- names(pathway_gene_sets)
n <- length(pathway_names)
similarity_matrix <- matrix(0, nrow = n, ncol = n, dimnames = list(pathway_names, pathway_names))

for (i in seq_len(n)) {
  for (j in seq_len(n)) {
    similarity_matrix[i, j] <- similarity_by_min(pathway_gene_sets[[i]], pathway_gene_sets[[j]])
  }
}

In [ ]:
ann_colors <- list(
    virus_type = setNames(colorRampPalette(brewer.pal(n_virus_type, "Set3"))(n_virus_type), unique(metadata_col$virus_type)),
    batch_id = setNames(colorRampPalette(brewer.pal(n_batch_id, "Dark2"))(n_batch_id), unique(metadata_col$batch_id)),
    condition = setNames(colorRampPalette(brewer.pal(n_condition, "Paired"))(n_condition), unique(metadata_col$condition))
)

my_colors <- colorRampPalette(c("navy", "white", "firebrick3"))(100)

reactome_heatmap_folder <- '/rprojectnb/cancergrp/brb/reproducibility/intermediate_files/GSEA_objects/heatmaps/reactome_simmilarity/'

if (!dir.exists(reactome_heatmap_folder)) {
  dir.create(reactome_heatmap_folder, recursive = TRUE)
}

## Plot heatmap
out <- pheatmap(similarity_matrix,
         clustering_distance_rows = "euclidean",
         cluster_rows = TRUE,
         # cluster_cols = FALSE,
         # breaks = seq(-rg, rg, length.out = 100),
         clustering_distance_cols = "euclidean",
         clustering_method = "complete",
         # labels_row = row_labels,
         # labels_col = col_labels,
         color = my_colors,
         show_rownames = FALSE,
         show_colnames = FALSE,
         # annotation_col = metadata_col[, vtr_annotations[[heatmap_j]], drop = FALSE],
         annotation_colors = ann_colors,
         border_color  = 'black',
         # main = heatmap_title,
         fontsize = 8,
         filename = paste0(reactome_heatmap_folder, "pathway_similarity_by_min.pdf"),
         width = 12,
         height = 10)

In [ ]:
# Initialize the clustered matrix with the correct number of rows
similarity_matrix.clust <- data.frame(row.names = rownames(similarity_matrix))

# Add clustering columns for k = 5 to k = 50 by steps of 5
for (k in seq(5, 50, by = 5)) {
  column_name <- paste0("k_", k)  # Create column name dynamically
  clusters <- cutree(out$tree_row, k = k)  # Generate clusters
  
  # Add the cluster column to the initialized data frame
  similarity_matrix.clust[[column_name]] <- clusters
}

# Reorder score_matrix2 and bind with the cluster matrix
reordered_matrix <- cbind(
  similarity_matrix[out$tree_row[["order"]], out$tree_col[["order"]]],  # Reordered original matrix
  similarity_matrix.clust[out$tree_row[["order"]], ]  # Reordered cluster matrix
)
            
write_xlsx(data.frame(Paths = rownames(reordered_matrix), reordered_matrix, check.names = FALSE), 
           paste0(reactome_heatmap_folder, "pathway_similarity_by_min.xlsx"))

In [ ]:
reactome_simmilarity_matrix <- read.xlsx(paste0(reactome_heatmap_folder, "pathway_similarity_by_min.xlsx"))

In [ ]:
paths <- reactome_simmilarity_matrix$Paths

# Create DataFrame
df <- data.frame(
  pathway = paths,
  genes = sapply(paths, function(p) {
    if (p %in% names(pathway_gene_sets)) {
      paste(pathway_gene_sets[[p]], collapse = ", ")
    } else {
      NA
    }
  }),
  stringsAsFactors = FALSE
)

write_xlsx(df, paste0(reactome_heatmap_folder, 'pathway_similarity_by_min_genes.xlsx'))

In [ ]:
# Function to filter columns based on vtr_intersect index
filter_columns <- function(data, vtr_intersect, index) {
  columns_to_keep <- colnames(data)[sapply(colnames(data), function(col) {
    all(str_detect(col, str_split(vtr_intersect[index], "_")[[1]]))
  })]
  return(data[, columns_to_keep, drop = FALSE])
}

In [ ]:
folder_path1 <- paste0("/rprojectnb/cancergrp/brb/reproducibility/intermediate_files/GSEA_objects/")

table_file <- paste0(folder_path1, "master_table.xlsx")
pvalue_file <- paste0(folder_path1, "pvalue_table.xlsx")
    
folder_path <- paste0("/rprojectnb/cancergrp/brb/reproducibility/intermediate_files/DGE_objects/heatmaps/L2FC/samples_vs_samples/")
vtr_df = read_excel('/rprojectnb/cancergrp/brb/raw_data/VTR_INFORMATION.xlsx')
vtr_codebook = read_excel('/rprojectnb/cancergrp/brb/raw_data/VTR_codebook.xlsx', sheet = 1)
xlsx_files <- list.files(path = "/rprojectnb/cancergrp/brb/reproducibility/intermediate_files/DGE_objects/filtered_tables/", pattern = 'strategy2', full.names = TRUE)

print(dim(combined_df))

L2FC_concordances_w_0 <- c()
L2FC_correlations_w_0 <- c()
L2FC_concordances_no_0 <- c()
L2FC_correlations_no_0 <- c()
NES_concordances_w_0 <- c()
NES_correlations_w_0 <- c()
NES_concordances_no_0 <- c()
NES_correlations_no_0 <- c()


for (m_index in 1:2) {

    condition_matrices <- list(w_0 = list(),
                                  no_0 = list())

    if (m_index == 1) {
        ### L2FC ###
        score_matrix_saved <- combined_df
        p_value_matrix <- pvalue_df
    
        # Make sure the column names of both matrices match before proceeding (excluding identifier columns)
        # Extract the shared part of the column names (everything before the last underscore)
        score_cols <- sub("_log2FC$", "", colnames(score_matrix_saved)[-1])
        pvalue_cols <- sub("_pvalue$", "", colnames(p_value_matrix)[-1])
        
        # Check if the relevant parts of the column names match
        if (!all(score_cols == pvalue_cols)) {
            stop("The column names of the score and p-value matrices do not match.")
        }
        
        # Exclude the "Paths" column from the numeric operations and retain it separately
        score_paths <- score_matrix_saved[,"symbol"]
        numeric_score_matrix <- score_matrix_saved[,-which(colnames(score_matrix_saved) == "symbol")]
        numeric_pvalue_matrix <- p_value_matrix[,-which(colnames(p_value_matrix) == "symbol")]
    
        numeric_score_matrix_w_0 <- numeric_score_matrix
    
        numeric_score_matrix_no_0 <- numeric_score_matrix
        
        # Set L2FC values to 0 where p-value is greater than 0.1
        numeric_score_matrix_w_0[numeric_pvalue_matrix > 0.05] <- 0

        for (condition_id in c("stimulated", "unstimulated")) {
        
            # Reattach the "Paths" column to the modified numeric matrix
            score_matrix_no_0 <- cbind(Paths = score_paths, numeric_score_matrix_no_0)
            score_matrix_w_0 <- cbind(Paths = score_paths, numeric_score_matrix_w_0)
    
            # Filter columns by condition (if necessary)
            score_matrix_no_0 <- score_matrix_no_0[, c("symbol", grep(paste0("_", condition_id), 
                                                                       colnames(score_matrix_no_0), 
                                                                       value = TRUE))]
            score_matrix_w_0 <- score_matrix_w_0[, c("symbol", grep(paste0("_", condition_id), 
                                                                       colnames(score_matrix_w_0), 
                                                                       value = TRUE))]

            condition_matrices$w_0[[condition_id]] <- score_matrix_w_0
            condition_matrices$no_0[[condition_id]] <- score_matrix_no_0

        }
        

    } else {
        
        ### NES ###
        score_matrix_saved <- read.xlsx(table_file)
        p_value_matrix <- read.xlsx(pvalue_file)

        # Make sure the column names of both matrices match before proceeding (excluding identifier columns)
        if (!all(colnames(score_matrix_saved)[-1] == colnames(p_value_matrix)[-1])) {
            stop("The column names of the score and p-value matrices do not match.")
        }
        
        # Exclude the "Paths" column from the numeric operations and retain it separately
        score_paths <- score_matrix_saved[,"Paths"]
        numeric_score_matrix <- score_matrix_saved[,-which(colnames(score_matrix_saved) == "Paths")]
        numeric_pvalue_matrix <- p_value_matrix[,-which(colnames(p_value_matrix) == "Paths")]
    
        numeric_score_matrix_w_0 <- numeric_score_matrix
    
        numeric_score_matrix_no_0 <- numeric_score_matrix

        for (condition_id in c("stimulated", "unstimulated")) {
        
            # Reattach the "Paths" column to the modified numeric matrix
            score_matrix_no_0 <- cbind(Paths = score_paths, numeric_score_matrix_no_0)
            score_matrix_w_0 <- cbind(Paths = score_paths, numeric_score_matrix_w_0)
    
            # Filter columns by condition (if necessary)
            score_matrix_no_0 <- score_matrix_no_0[, c("Paths", grep(paste0("_", condition_id), 
                                                                       colnames(score_matrix_no_0), 
                                                                       value = TRUE))]
            score_matrix_w_0 <- score_matrix_w_0[, c("Paths", grep(paste0("_", condition_id), 
                                                                       colnames(score_matrix_w_0), 
                                                                       value = TRUE))]

            condition_matrices$w_0[[condition_id]] <- score_matrix_w_0
            condition_matrices$no_0[[condition_id]] <- score_matrix_no_0

        }

    }
    
    
    vtr_intersect <- intersect(paste(str_split_fixed(colnames(condition_matrices$w_0[['stimulated']])[-1], '\\_', 4)[, 1],
                      str_split_fixed(colnames(condition_matrices$w_0[['stimulated']])[-1], '\\_', 4)[, 3], sep="_"),
                    paste(str_split_fixed(colnames(condition_matrices$w_0[['unstimulated']])[-1], '\\_', 4)[, 1],
                      str_split_fixed(colnames(condition_matrices$w_0[['unstimulated']])[-1], '\\_', 4)[, 3], sep="_"))

    print(length(vtr_intersect))
    
    for (i in 1:length(vtr_intersect)) {

        for (matrix_i in 1:2) {

            stim <- condition_matrices[[matrix_i]][['stimulated']]
            unstim <- condition_matrices[[matrix_i]][['unstimulated']]
    
            # Filter columns from stim and unstim based on vtr_intersect[46]
            sub_stim <- filter_columns(stim, vtr_intersect, i)
            sub_unstim <- filter_columns(unstim, vtr_intersect, i)
    
            if (m_index == 1) {
                 # Ensure that the Paths column is retained for merging
                sub_stim$symbol <- stim$symbol[match(rownames(sub_stim), rownames(stim))]
                sub_unstim$symbol <- unstim$symbol[match(rownames(sub_unstim), rownames(unstim))]
                
                # Merge the two data frames on the Paths column
                merged_data <- merge(sub_stim, sub_unstim, by = "symbol", suffixes = c("_stim", "_unstim"))
                rownames(merged_data) <- merged_data$symbol
                
            } else {
                # Ensure that the Paths column is retained for merging
                sub_stim$Paths <- stim$Paths[match(rownames(sub_stim), rownames(stim))]
                sub_unstim$Paths <- unstim$Paths[match(rownames(sub_unstim), rownames(unstim))]
                
                # Merge the two data frames on the Paths column
                merged_data <- merge(sub_stim, sub_unstim, by = "Paths", suffixes = c("_stim", "_unstim"))
            }
        
            # Check the dimensions of the merged data
            dim(merged_data)
            
            # Specify the names of the columns to plot
            x_col <- colnames(merged_data)[2]  # Second column
            y_col <- colnames(merged_data)[3]  # Third column
            
            # Extract the values for the selected columns
            x_values_raw <- merged_data[[x_col]]
            names(x_values_raw) <- rownames(merged_data)
            y_values_raw <- merged_data[[y_col]]
            names(y_values_raw) <- rownames(merged_data)
            
            x_values <- x_values_raw
            y_values <- y_values_raw

            ns_values_x <- rep(FALSE, length(score_paths))
            ns_values_y <- rep(FALSE, length(score_paths))

            # Match score_path with score_matrix Paths to make sure they're aligned
            ns_values_x[match(score_paths, merged_data[1])
                        [which(numeric_pvalue_matrix[, gsub('log2FC', 'pvalue', x_col)] > 0.05)]] <- TRUE
            ns_values_y[match(score_paths, merged_data[1])
                        [which(numeric_pvalue_matrix[, gsub('log2FC', 'pvalue', y_col)] > 0.05)]] <- TRUE

            if (matrix_i == 1) {
                # Set NA where both x and y are 0 (to ignore these pairs)
                x_values[x_values == 0 & y_values == 0] <- NA
                y_values[x_values == 0 & y_values == 0] <- NA

            } else {
                # Set NA where both x and y are 0 (to ignore these pairs)
                x_values[ns_values_x & ns_values_y] <- NA
                y_values[ns_values_x & ns_values_y] <- NA
            }

            ##### SCATTER PLOT ####
            # Create a logical index for non-NA and non-zero values in x and y
            non_na_idx <- !is.na(x_values) & !is.na(y_values)

            # Create a filtered data frame for ggplot2
            filtered_data <- data.frame(x_val = x_values[non_na_idx], y_val = y_values[non_na_idx])

            plot_title <- ifelse(m_index == 1,
                                paste("L2FC: ", paste(strsplit(x_col, "_")[[1]][1:3], collapse = "_"), " vs ",
                                      paste(strsplit(y_col, "_")[[1]][1:3], collapse = "_")),
                                paste("NES: ", vtr_intersect[i], " Stim vs Unstim"))
            
            # Create the scatter plot
            p <- ggplot(filtered_data, aes(x = x_val, y = y_val)) +
                geom_point() +
                theme_bw() +
                geom_abline(slope = 1, intercept = 0, color = "blue") +
                stat_cor(method = "pearson", label.x = -1.3, label.y = 1.2, size = 5) +
                ggtitle(plot_title)
    
            plot_folder <- ifelse(m_index == 1,
                                  paste0("/rprojectnb/cancergrp/brb/reproducibility/intermediate_files/DGE_objects/scatter_plot/L2FC/"),
                                  paste0("/rprojectnb/cancergrp/brb/reproducibility/intermediate_files/GSEA_objects/scatter_plot/NES/"))
            dir.create(plot_folder, recursive = TRUE, showWarnings = FALSE)
            
            if (matrix_i == 1) {
                plot_file <- ifelse(m_index == 1,
                                    paste0(plot_folder, paste(strsplit(x_col, "_")[[1]][1:3], collapse = "_"), "_vs_",
                                    paste(strsplit(y_col, "_")[[1]][1:3], collapse = "_"), "_w_0.pdf"),
                                    paste0(plot_folder, "scatter_plot_", vtr_intersect[i], "_w_0.pdf"))
            } else {

                plot_file <- ifelse(m_index == 1,
                                    paste0(plot_folder, paste(strsplit(x_col, "_")[[1]][1:3], collapse = "_"), "_vs_",
                                    paste(strsplit(y_col, "_")[[1]][1:3], collapse = "_"), "_no_0.pdf"),
                                    paste0(plot_folder, "scatter_plot_", vtr_intersect[i], "_no_0.pdf"))
                
            }

            ggsave(filename = plot_file, plot = p)

            ##### CONCORDANCE ####
            
            # Determine C and D values
            C <- sum((x_values > 0 & y_values > 0) | (x_values < 0 & y_values < 0), na.rm = TRUE) # Same sign count
            D <- sum((x_values > 0 & y_values < 0) | (x_values < 0 & y_values > 0) |
                     (x_values == 0 & y_values != 0) | (x_values != 0 & y_values == 0), na.rm = TRUE) # Different sign or zero pairs (excluding both-zero pairs)
            
            score_all <- (C - D) / (C + D)
            
            #### PEARSON CORRELATION ####
            
            # Calculate correlation for all pairs (excluding rows with NA values)
            correlation_all <- cor(x_values[non_na_idx], y_values[non_na_idx], use = "pairwise.complete.obs")
            
            # print(vtr_intersect[i])
            if (m_index == 1) {
                if (matrix_i == 1) {
                    L2FC_concordances_w_0 <- c(L2FC_concordances_w_0, setNames(score_all, vtr_intersect[i]))
                    L2FC_correlations_w_0 <- c(L2FC_correlations_w_0, setNames(correlation_all, vtr_intersect[i]))
                } else {
                    L2FC_concordances_no_0 <- c(L2FC_concordances_no_0, setNames(score_all, vtr_intersect[i]))
                    L2FC_correlations_no_0 <- c(L2FC_correlations_no_0, setNames(correlation_all, vtr_intersect[i]))
                }
            } else {
                if (matrix_i == 1) {
                    NES_concordances_w_0 <- c(NES_concordances_w_0, setNames(score_all, vtr_intersect[i]))
                    NES_correlations_w_0 <- c(NES_correlations_w_0, setNames(correlation_all, vtr_intersect[i]))
                } else {
                    NES_concordances_no_0 <- c(NES_concordances_no_0, setNames(score_all, vtr_intersect[i]))
                    NES_correlations_no_0 <- c(NES_correlations_no_0, setNames(correlation_all, vtr_intersect[i]))
                }
            }
            # break
        }
        #break
    }
    # break
}

concordance_summary <- data.frame(VTR = names(NES_concordances_w_0),
                                  NES_concordance_no_0 = NES_concordances_no_0,
                                  L2FC_concordance_no_0 = L2FC_concordances_no_0[names(NES_concordances_w_0)],
                                 NES_concordance_w_0 = NES_concordances_w_0[names(NES_concordances_w_0)],
                                 L2FC_concordance_w_0 = L2FC_concordances_w_0[names(NES_concordances_w_0)],
                                 NES_correlation_no_0 = NES_correlations_no_0[names(NES_concordances_w_0)],
                                  L2FC_correlation_no_0 = L2FC_correlations_no_0[names(NES_concordances_w_0)],
                                 NES_correlation_w_0 = NES_correlations_w_0[names(NES_concordances_w_0)],
                                 L2FC_correlation_w_0 = L2FC_correlations_w_0[names(NES_concordances_w_0)])

write.xlsx(concordance_summary, paste0("/rprojectnb/cancergrp/brb/reproducibility/intermediate_files/concordance_correlation_summary.xlsx"))

#### NES

In [ ]:
vtr_df = read_excel('/rprojectnb/cancergrp/brb/raw_data/VTR_INFORMATION.xlsx')
vtr_codebook = read_excel('/rprojectnb/cancergrp/brb/raw_data/VTR_codebook.xlsx', sheet = 1)

folder_path1 <- paste0("/rprojectnb/cancergrp/brb/reproducibility/intermediate_files/GSEA_objects/")
table_file <- paste0(folder_path1, "master_table.xlsx")
pvalue_file <- paste0(folder_path1, "pvalue_table.xlsx")

score_matrix_saved <- read.xlsx(table_file)
p_value_matrix <- read.xlsx(pvalue_file)
         
# Make sure the column names of both matrices match before proceeding (excluding identifier columns)
if (!all(colnames(score_matrix_saved)[-1] == colnames(p_value_matrix)[-1])) {
    stop("The column names of the score and p-value matrices do not match.")
}

# Exclude the "Paths" column from the numeric operations and retain it separately
score_paths <- score_matrix_saved[,"Paths"]
numeric_score_matrix <- score_matrix_saved[,-which(colnames(score_matrix_saved) == "Paths")]
numeric_pvalue_matrix <- p_value_matrix[,-which(colnames(p_value_matrix) == "Paths")]

numeric_score_matrix_w_0 <- numeric_score_matrix

numeric_score_matrix_no_0 <- numeric_score_matrix

# Set NES values to 0 where p-value is greater than 0.1
numeric_score_matrix_w_0[numeric_pvalue_matrix > 0.05] <- 0

# Reattach the "Paths" column to the modified numeric matrix
score_matrix_no_0 <- cbind(Paths = score_paths, numeric_score_matrix_no_0)
score_matrix_w_0 <- cbind(Paths = score_paths, numeric_score_matrix_w_0)

# Split the first element of the vector into two parts
split_results <- str_split_fixed(vtr_intersect, "_", 2)
first_values <- split_results[,1]
last_values <- split_results[,2]

stim_cols <-  colnames(score_matrix_no_0)[grep(paste0("_", "stimulated"), colnames(score_matrix_no_0))]
unstim_cols <- colnames(score_matrix_no_0)[grep(paste0("_", "unstimulated"), colnames(score_matrix_no_0))]

# Filter items that start with `first_val` and end with `last_val`
stim_cols <- stim_cols[
  sapply(stim_cols, function(x) {
    parts <- str_split_fixed(x, "_", 3)
    parts[1] %in% first_values && parts[3] %in% last_values
  })
]

unstim_cols <- unstim_cols[
  sapply(unstim_cols, function(x) {
    parts <- str_split_fixed(x, "_", 3)
    parts[1] %in% first_values && parts[3] %in% last_values
  })
]

# Filter columns by condition (if necessary)
score_matrix_no_0_stim <- score_matrix_no_0[, c("Paths", stim_cols)]
score_matrix_no_0_unstim <- score_matrix_no_0[, c("Paths", unstim_cols)]
score_matrix_w_0_stim <- score_matrix_w_0[, c("Paths", stim_cols)]
score_matrix_w_0_unstim <- score_matrix_w_0[, c("Paths", unstim_cols)]
    
# Exclude the first column if it’s an identifier and keep only numeric columns
data_matrix_no_0_stim <- as.matrix(score_matrix_no_0_stim[,-1])
data_matrix_w_0_stim <- as.matrix(score_matrix_w_0_stim[,-1])
data_matrix_no_0_unstim <- as.matrix(score_matrix_no_0_unstim[,-1])
data_matrix_w_0_unstim <- as.matrix(score_matrix_w_0_unstim[,-1])

matrices_0 <- list(with_0 = list(stim = data_matrix_w_0_stim, unstim = data_matrix_w_0_unstim),
                   no_0 = list(stim = data_matrix_no_0_stim, unstim = data_matrix_no_0_unstim))

# Create empty Pearson correlation allData
empty_matrix <- matrix(NA, ncol = ncol(data_matrix_no_0_stim), nrow = ncol(data_matrix_no_0_unstim))
colnames(empty_matrix) <- colnames(data_matrix_no_0_stim)
rownames(empty_matrix) <- colnames(data_matrix_no_0_unstim)

cor_matrix_w_0 <- empty_matrix
cor_matrix_no_0 <- empty_matrix
concord_matrix_w_0 <- empty_matrix
concord_matrix_no_0 <- empty_matrix

result_matrices <- list(correlation = list(with_0 = cor_matrix_w_0, no_0 = cor_matrix_no_0),
                       concordance = list(with_0 = concord_matrix_w_0, no_0 = concord_matrix_no_0))

# Calculate pairwise correlations
all_combinations <- expand.grid(colnames(data_matrix_no_0_stim), colnames(data_matrix_no_0_unstim))
for (pair_ix in 1:nrow(all_combinations)) {

    for (matrix_i in 1:length(matrices_0)) {
        
        pair <- all_combinations[pair_ix, ]
        x_col <- pair[1] %>% pull()
        y_col <- pair[2] %>% pull()
        
        # Extract the column values
        x_values <- matrices_0[[matrix_i]][['stim']][, x_col]
        y_values <- matrices_0[[matrix_i]][['unstim']][, y_col]

        ns_values_x <- rep(FALSE, length(score_paths))
        ns_values_y <- rep(FALSE, length(score_paths))

        # Match score_path with score_matrix Paths to make sure they're aligned
        ns_values_x[match(score_paths, score_matrix_no_0_stim$Paths)[which(numeric_pvalue_matrix[, x_col] > 0.05)]] <- TRUE
        ns_values_y[match(score_paths, score_matrix_no_0_unstim$Paths)[which(numeric_pvalue_matrix[, y_col] > 0.05)]] <- TRUE

        if (matrix_i == 1) {
            # Set NA where both x and y are 0 (to ignore these pairs)
            x_values[x_values == 0 & y_values == 0] <- NA
            y_values[x_values == 0 & y_values == 0] <- NA

        } else {
            # Set NA where both x and y are non significant (to ignore these pairs)
            x_values[ns_values_x & ns_values_y] <- NA
            y_values[ns_values_x & ns_values_y] <- NA
            
        }

        ##### CONCORDANCE ####
        
        # Determine C and D values
        C <- sum((x_values > 0 & y_values > 0) | (x_values < 0 & y_values < 0), na.rm = TRUE) # Same sign count
        D <- sum((x_values > 0 & y_values < 0) | (x_values < 0 & y_values > 0) |
                 (x_values == 0 & y_values != 0) | (x_values != 0 & y_values == 0), na.rm = TRUE) # Different sign or zero pairs (excluding both-zero pairs)
        
        score_all <- (C - D) / (C + D)
        
        #### PEARSON CORRELATION ####

        # Create a logical index for non-NA and non-zero values in x and y
        non_na_idx <- !is.na(x_values) & !is.na(y_values)
        
        # Calculate correlation for all pairs (excluding rows with NA values)
        correlation_all <- cor(x_values[non_na_idx], y_values[non_na_idx], use = "pairwise.complete.obs")
        
        # Store the correlation/concordance coefficient in the matrix (symmetric)

        result_matrices$correlation[[matrix_i]][x_col, y_col] <- correlation_all
        result_matrices$concordance[[matrix_i]][x_col, y_col] <- score_all

    }
    
}

In [ ]:
folder_path <- paste0("/rprojectnb/cancergrp/brb/reproducibility/intermediate_files/GSEA_objects/heatmaps/NES/stim_vs_unstim/")

for (heatmap_i in 1:length(result_matrices)) {

    for (matrix_i in 1:length(result_matrices[[heatmap_i]])) {

        metadata_col <- data.frame(id = c(colnames(result_matrices[[heatmap_i]][[matrix_i]]),
                                         rownames(result_matrices[[heatmap_i]][[matrix_i]])))
        metadata_col$batch_id = str_split_fixed(metadata_col$id, '\\_', 4)[,3]
        metadata_col$condition = str_split_fixed(metadata_col$id, '\\_', 4)[,2]
        metadata_col$vtr = str_split_fixed(metadata_col$id, '\\_', 4)[,1]
        # metadata_col$virus_type = vtr_df$`Viral family`[match(metadata_col$vtr, vtr_df$code)]
        metadata_col$virus_type <- ifelse(
                                          metadata_col$vtr == "ctrl", 
                                          "ctrl", 
                                          vtr_df$`Viral family`[match(metadata_col$vtr, vtr_df$code)]
                                        )
        rownames(metadata_col) = metadata_col$id
        # Map `Virus` from `vtr_df` based on `vtr` code
         metadata_col$virus_name <- ifelse(
            metadata_col$vtr == "ctrl", 
            "ctrl", 
            vtr_codebook$`Virus`[match(metadata_col$vtr, vtr_codebook$code)]
          )
          
          # Create the new combined labels (e.g., "VTR10 (Virus)")
          metadata_col$combined_label <- ifelse(
            metadata_col$vtr == "ctrl", 
            "ctrl", 
            paste0(metadata_col$vtr, " (", metadata_col$virus_name, ")")
          )
        # Assign the combined labels to row and column names for the heatmap
          # row_labels <- metadata_col$combined_label
          # col_labels <- metadata_col$combined_label
        
        # Crete palette colors
        n_virus_type = length(unique(metadata_col$virus_type))
        n_batch_id = length(unique(metadata_col$batch_id))
        n_condition = length(unique(metadata_col$condition))
        
        ann_colors <- list(
            virus_type = setNames(colorRampPalette(brewer.pal(n_virus_type, "Set3"))(n_virus_type), unique(metadata_col$virus_type)),
            batch_id = setNames(colorRampPalette(brewer.pal(n_batch_id, "Dark2"))(n_batch_id), unique(metadata_col$batch_id)),
            condition = setNames(colorRampPalette(brewer.pal(n_condition, "Paired"))(n_condition), unique(metadata_col$condition))
        )
        my_colors <- colorRampPalette(c("navy", "white", "firebrick3"))(100)

        if (heatmap_i == 1) {
            if (matrix_i == 1) {
                heatmap_folder <- paste0(folder_path, "correlation")
                heatmap_file <- paste0(heatmap_folder, "/nes_correlation_heatmap_w_0")
                heatmap_title <- "Correlation Heatmap"
            } else {
                heatmap_folder <- paste0(folder_path, "correlation")
                heatmap_title <- "Correlation Heatmap"
                heatmap_file <- paste0(heatmap_folder, "/nes_correlation_heatmap_no_0")
            }
        } else if (heatmap_i == 2) {
            if (matrix_i == 1) {
                heatmap_folder <- paste0(folder_path, "concordance")
                heatmap_title <- "Concordance Heatmap"
                heatmap_file <- paste0(heatmap_folder, "/nes_concordance_heatmap_w_0")
            } else {
                heatmap_folder <- paste0(folder_path, "concordance")
                heatmap_title <- "Concordance Heatmap"
                heatmap_file <- paste0(heatmap_folder, "/nes_concordance_heatmap_no_0")
            }
        }

        heatmap_file <- paste0(heatmap_file)

        if (!dir.exists(heatmap_folder)) {
          dir.create(heatmap_folder, recursive = TRUE)
        }
        
        ## Specify breaks
        rg <- max(abs(result_matrices[[heatmap_i]][[matrix_i]]))

        if (!is.finite(rg)) {
            rg <- 1  # Set rg to a fixed value (can use 1 for the upper limit)
        }

        vtr_annotations <- list(c('condition', 'virus_type', 'batch_id'), c('virus_type'))
        
        for (heatmap_j in 1:length(vtr_annotations)) {
            
            ## Plot heatmap
            out <- pheatmap(result_matrices[[heatmap_i]][[matrix_i]],
                     clustering_distance_rows = "euclidean",
                     cluster_rows = TRUE,
                     # cluster_cols = FALSE,
                     breaks = seq(-rg, rg, length.out = 100),
                     clustering_distance_cols = "euclidean",
                     clustering_method = "complete",
                     # labels_row = row_labels,
                     # labels_col = col_labels,
                     color = my_colors,
                     show_rownames = FALSE,
                     show_colnames = FALSE,
                     annotation_col = metadata_col[, vtr_annotations[[heatmap_j]], drop = FALSE],
                     annotation_row = metadata_col[, vtr_annotations[[heatmap_j]], drop = FALSE],
                     annotation_colors = ann_colors,
                     border_color  = 'black',
                     main = heatmap_title,
                     fontsize = 8,
                     filename = paste0(heatmap_file, "_", heatmap_j, ".pdf"),
                     width = 12,
                     height = 10)
        }

        reordered_matrix <- result_matrices[[heatmap_i]][[matrix_i]][out$tree_row[["order"]], out$tree_col[["order"]]]
        
        write_xlsx(data.frame(VTR = rownames(reordered_matrix), reordered_matrix, check.names = FALSE), paste0(heatmap_file, ".xlsx"))
        #break
    }
    #break
}

#### L2FC

In [ ]:
# Create empty data frames to store log2FoldChange and pvalue values separately
combined_df <- data.frame()
pvalue_df <- data.frame()

# Loop through each file and extract log2FoldChange and pvalue columns
for (file in xlsx_files) {
    # Read the file into a data frame
    df <- read_xlsx(file)
    
    # Check if the required columns exist in the file (e.g., symbol, log2FoldChange, and pvalue)
    if ("log2FoldChange" %in% colnames(df) & "symbol" %in% colnames(df) & "pvalue" %in% colnames(df)) {
        
        # Extract the base name of the file to use as a prefix for the columns
        file_name <- file_path_sans_ext(basename(file))
        
        # Extract and rename log2FoldChange column
        temp_log2FC_df <- df %>%
            select(symbol, log2FoldChange) %>%
            rename(!!paste0(file_name, "_log2FC") := log2FoldChange)
        
        # Extract and rename pvalue column
        temp_pvalue_df <- df %>%
            select(symbol, pvalue) %>%
            rename(!!paste0(file_name, "_pvalue") := pvalue)
        
        # Merge log2FoldChange values with combined_df by 'symbol'
        if (nrow(combined_df) == 0) {
            combined_df <- temp_log2FC_df
        } else {
            combined_df <- full_join(combined_df, temp_log2FC_df, by = "symbol")
        }

        # Merge pvalue values with pvalue_df by 'symbol'
        if (nrow(pvalue_df) == 0) {
            pvalue_df <- temp_pvalue_df
        } else {
            pvalue_df <- full_join(pvalue_df, temp_pvalue_df, by = "symbol")
        }
    }
}

In [ ]:
vtr_df = read_excel('/rprojectnb/cancergrp/brb/raw_data/VTR_INFORMATION.xlsx')
vtr_codebook = read_excel('/rprojectnb/cancergrp/brb/raw_data/VTR_codebook.xlsx', sheet = 1)
xlsx_files <- list.files(path = "/rprojectnb/cancergrp/brb/reproducibility/intermediate_files/DGE_objects/filtered_tables/", pattern = 'strategy2', full.names = TRUE)

# print(xlsx_files)

# Initialize an empty data frame to store log2FoldChange values
combined_df <- data.frame()
pvalue_df <- data.frame()

# Loop through each file and extract log2FoldChange and pvalue columns
for (file in xlsx_files) {
    # Read the file into a data frame
    df <- read_xlsx(file)
    
    # Check if the required columns exist in the file (e.g., symbol, log2FoldChange, and pvalue)
    if ("log2FoldChange" %in% colnames(df) & "symbol" %in% colnames(df) & "pvalue" %in% colnames(df)) {
        
        # Extract the base name of the file to use as a prefix for the columns
        file_name <- file_path_sans_ext(basename(file))
        
        # Extract and rename log2FoldChange column
        temp_log2FC_df <- df %>%
            select(symbol, log2FoldChange) %>%
            rename(!!paste0(file_name, "_log2FC") := log2FoldChange)
        
        # Extract and rename pvalue column
        temp_pvalue_df <- df %>%
            select(symbol, pvalue) %>%
            rename(!!paste0(file_name, "_pvalue") := pvalue)
        
        # Merge log2FoldChange values with combined_df by 'symbol'
        if (nrow(combined_df) == 0) {
            combined_df <- temp_log2FC_df
        } else {
            combined_df <- full_join(combined_df, temp_log2FC_df, by = "symbol")
        }

        # Merge pvalue values with pvalue_df by 'symbol'
        if (nrow(pvalue_df) == 0) {
            pvalue_df <- temp_pvalue_df
        } else {
            pvalue_df <- full_join(pvalue_df, temp_pvalue_df, by = "symbol")
        }
    }
}

print(dim(combined_df))

# rownames(combined_df) <- combined_df$symbol

score_matrix_saved <- combined_df
p_value_matrix <- pvalue_df

# Make sure the column names of both matrices match before proceeding (excluding identifier columns)
# Extract the shared part of the column names (everything before the last underscore)
score_cols <- sub("_log2FC$", "", colnames(score_matrix_saved)[-1])
pvalue_cols <- sub("_pvalue$", "", colnames(p_value_matrix)[-1])

# Check if the relevant parts of the column names match
if (!all(score_cols == pvalue_cols)) {
    stop("The column names of the score and p-value matrices do not match.")
}

# Exclude the "Paths" column from the numeric operations and retain it separately
score_paths <- score_matrix_saved[,"symbol"]
numeric_score_matrix <- score_matrix_saved[,-which(colnames(score_matrix_saved) == "symbol")]
numeric_pvalue_matrix <- p_value_matrix[,-which(colnames(p_value_matrix) == "symbol")]

numeric_score_matrix_w_0 <- numeric_score_matrix

numeric_score_matrix_no_0 <- numeric_score_matrix

# Set L2FC values to 0 where p-value is greater than 0.1
numeric_score_matrix_w_0[numeric_pvalue_matrix > 0.05] <- 0

# Reattach the "Paths" column to the modified numeric matrix
score_matrix_no_0 <- cbind(Paths = score_paths, numeric_score_matrix_no_0)
score_matrix_w_0 <- cbind(Paths = score_paths, numeric_score_matrix_w_0)


# Split the first element of the vector into two parts
split_results <- str_split_fixed(vtr_intersect, "_", 2)
first_values <- split_results[,1]
last_values <- split_results[,2]

stim_cols <-  colnames(score_matrix_no_0)[grep(paste0("_", "stimulated"), colnames(score_matrix_no_0))]
unstim_cols <- colnames(score_matrix_no_0)[grep(paste0("_", "unstimulated"), colnames(score_matrix_no_0))]

# Filter items that start with `first_val` and end with `last_val`
stim_cols <- stim_cols[
  sapply(stim_cols, function(x) {
    parts <- str_split_fixed(x, "_", 4)
    parts[1] %in% first_values && parts[3] %in% last_values
  })
]

unstim_cols <- unstim_cols[
  sapply(unstim_cols, function(x) {
    parts <- str_split_fixed(x, "_", 4)
    parts[1] %in% first_values && parts[3] %in% last_values
  })
]

# Filter columns by condition (if necessary)
score_matrix_no_0_stim <- score_matrix_no_0[, c("symbol", stim_cols)]
score_matrix_no_0_unstim <- score_matrix_no_0[, c("symbol", unstim_cols)]
score_matrix_w_0_stim <- score_matrix_w_0[, c("symbol", stim_cols)]
score_matrix_w_0_unstim <- score_matrix_w_0[, c("symbol", unstim_cols)]
    
# Exclude the first column if it’s an identifier and keep only numeric columns
data_matrix_no_0_stim <- as.matrix(score_matrix_no_0_stim[,-1])
data_matrix_w_0_stim <- as.matrix(score_matrix_w_0_stim[,-1])
data_matrix_no_0_unstim <- as.matrix(score_matrix_no_0_unstim[,-1])
data_matrix_w_0_unstim <- as.matrix(score_matrix_w_0_unstim[,-1])

matrices_0 <- list(with_0 = list(stim = data_matrix_w_0_stim, unstim = data_matrix_w_0_unstim),
                   no_0 = list(stim = data_matrix_no_0_stim, unstim = data_matrix_no_0_unstim))

# Exclude the first column if it’s an identifier and keep only numeric columns
data_matrix_no_0 <- as.matrix(score_matrix_no_0[,-1])
data_matrix_w_0 <- as.matrix(score_matrix_w_0[,-1])

matrices_0 <- list(with_0 = list(stim = data_matrix_w_0_stim, unstim = data_matrix_w_0_unstim),
                   no_0 = list(stim = data_matrix_no_0_stim, unstim = data_matrix_no_0_unstim))

# Create empty Pearson correlation allData
empty_matrix <- matrix(NA, ncol = ncol(data_matrix_no_0_stim), nrow = ncol(data_matrix_no_0_unstim))
colnames(empty_matrix) <- colnames(data_matrix_no_0_stim)
rownames(empty_matrix) <- colnames(data_matrix_no_0_unstim)

cor_matrix_w_0 <- empty_matrix
cor_matrix_no_0 <- empty_matrix
concord_matrix_w_0 <- empty_matrix
concord_matrix_no_0 <- empty_matrix

result_matrices <- list(correlation = list(with_0 = cor_matrix_w_0, no_0 = cor_matrix_no_0),
                       concordance = list(with_0 = concord_matrix_w_0, no_0 = concord_matrix_no_0))

# Calculate pairwise correlations
all_combinations <- expand.grid(colnames(data_matrix_no_0_stim), colnames(data_matrix_no_0_unstim))
for (pair_ix in 1:nrow(all_combinations)) {

    for (matrix_i in 1:length(matrices_0)) {
        
        pair <- all_combinations[pair_ix, ]
        x_col <- pair[1] %>% pull()
        y_col <- pair[2] %>% pull()
        
        # Extract the column values
        x_values <- matrices_0[[matrix_i]][['stim']][, x_col]
        y_values <- matrices_0[[matrix_i]][['unstim']][, y_col]

        ns_values_x <- rep(FALSE, length(score_paths))
        ns_values_y <- rep(FALSE, length(score_paths))

        # Match score_path with score_matrix Paths to make sure they're aligned
        ns_values_x[match(score_paths, score_matrix_no_0$symbol)
                    [which(numeric_pvalue_matrix[, gsub('log2FC', 'pvalue', x_col)] > 0.05)]] <- TRUE
        ns_values_y[match(score_paths, score_matrix_no_0$symbol)
                    [which(numeric_pvalue_matrix[, gsub('log2FC', 'pvalue', y_col)] > 0.05)]] <- TRUE
        

        if (matrix_i == 1) {
            # Set NA where both x and y are 0 (to ignore these pairs)
            x_values[x_values == 0 & y_values == 0] <- NA
            y_values[x_values == 0 & y_values == 0] <- NA

        } else {
            # Set NA where both x and y are 0 (to ignore these pairs)
            x_values[ns_values_x & ns_values_y] <- NA
            y_values[ns_values_x & ns_values_y] <- NA

        }

        ##### CONCORDANCE ####
        
        # Determine C and D values
        C <- sum((x_values > 0 & y_values > 0) | (x_values < 0 & y_values < 0), na.rm = TRUE) # Same sign count
        D <- sum((x_values > 0 & y_values < 0) | (x_values < 0 & y_values > 0) |
                 (x_values == 0 & y_values != 0) | (x_values != 0 & y_values == 0), na.rm = TRUE) # Different sign or zero pairs (excluding both-zero pairs)
        
        score_all <- (C - D) / (C + D)
        
        #### PEARSON CORRELATION ####

        # Create a logical index for non-NA and non-zero values in x and y
        non_na_idx <- !is.na(x_values) & !is.na(y_values)
        
        # Calculate correlation for all pairs (excluding rows with NA values)
        correlation_all <- cor(x_values[non_na_idx], y_values[non_na_idx], use = "pairwise.complete.obs")
        
        # Store the correlation/concordance coefficient in the matrix (symmetric)

        result_matrices$correlation[[matrix_i]][x_col, y_col] <- correlation_all
        result_matrices$concordance[[matrix_i]][x_col, y_col] <- score_all
    }
    
}

In [ ]:
folder_path <- paste0("/rprojectnb/cancergrp/brb/reproducibility/intermediate_files/DGE_objects/heatmaps/L2FC/stim_vs_unstim/")

for (heatmap_i in 1:length(result_matrices)) {

    for (matrix_i in 1:length(result_matrices[[heatmap_i]])) {

        metadata_col <- data.frame(id = c(colnames(result_matrices[[heatmap_i]][[matrix_i]]),
                                         rownames(result_matrices[[heatmap_i]][[matrix_i]])))
        metadata_col$batch_id = str_split_fixed(metadata_col$id, '\\_', 4)[,3]
        metadata_col$condition = str_split_fixed(metadata_col$id, '\\_', 4)[,2]
        metadata_col$vtr = str_split_fixed(metadata_col$id, '\\_', 4)[,1]
        # metadata_col$virus_type = vtr_df$`Viral family`[match(metadata_col$vtr, vtr_df$code)]
        metadata_col$virus_type <- ifelse(
                                          metadata_col$vtr == "ctrl", 
                                          "ctrl", 
                                          vtr_df$`Viral family`[match(metadata_col$vtr, vtr_df$code)]
                                        )
        rownames(metadata_col) = metadata_col$id
        # Map `Virus` from `vtr_df` based on `vtr` code
         metadata_col$virus_name <- ifelse(
            metadata_col$vtr == "ctrl", 
            "ctrl", 
            vtr_codebook$`Virus`[match(metadata_col$vtr, vtr_codebook$code)]
          )
          
          # Create the new combined labels (e.g., "VTR10 (Virus)")
          metadata_col$combined_label <- ifelse(
            metadata_col$vtr == "ctrl", 
            "ctrl", 
            paste0(metadata_col$vtr, " (", metadata_col$virus_name, ")")
          )
        # Assign the combined labels to row and column names for the heatmap
          # row_labels <- metadata_col$combined_label
          # col_labels <- metadata_col$combined_label
        
        # Crete palette colors
        n_virus_type = length(unique(metadata_col$virus_type))
        n_batch_id = length(unique(metadata_col$batch_id))
        n_condition = length(unique(metadata_col$condition))
        
        ann_colors <- list(
            virus_type = setNames(colorRampPalette(brewer.pal(n_virus_type, "Set3"))(n_virus_type), unique(metadata_col$virus_type)),
            batch_id = setNames(colorRampPalette(brewer.pal(n_batch_id, "Dark2"))(n_batch_id), unique(metadata_col$batch_id)),
            condition = setNames(colorRampPalette(brewer.pal(n_condition, "Paired"))(n_condition), unique(metadata_col$condition))
        )
        my_colors <- colorRampPalette(c("navy", "white", "firebrick3"))(100)

        if (heatmap_i == 1) {
            if (matrix_i == 1) {
                heatmap_folder <- paste0(folder_path, "correlation")
                heatmap_file <- paste0(heatmap_folder, "/l2fc_correlation_heatmap_w_0")
                heatmap_title <- "Correlation Heatmap"
            } else {
                heatmap_folder <- paste0(folder_path, "correlation")
                heatmap_title <- "Correlation Heatmap"
                heatmap_file <- paste0(heatmap_folder, "/l2fc_correlation_heatmap_no_0")
            }
        } else if (heatmap_i == 2) {
            if (matrix_i == 1) {
                heatmap_folder <- paste0(folder_path, "concordance")
                heatmap_title <- "Concordance Heatmap"
                heatmap_file <- paste0(heatmap_folder, "/l2fc_concordance_heatmap_w_0")
            } else {
                heatmap_folder <- paste0(folder_path, "concordance")
                heatmap_title <- "Concordance Heatmap"
                heatmap_file <- paste0(heatmap_folder, "/l2fc_concordance_heatmap_no_0")
            }
        }

        heatmap_file <- paste0(heatmap_file)

        if (!dir.exists(heatmap_folder)) {
          dir.create(heatmap_folder, recursive = TRUE)
        }
        
        ## Specify breaks
        rg <- max(abs(result_matrices[[heatmap_i]][[matrix_i]]))

        if (!is.finite(rg)) {
            rg <- 1  # Set rg to a fixed value (can use 1 for the upper limit)
        }

        vtr_annotations <- list(c('condition', 'virus_type', 'batch_id'), c('virus_type'))
        
        for (heatmap_j in 1:length(vtr_annotations)) {
            
            ## Plot heatmap
            out <- pheatmap(result_matrices[[heatmap_i]][[matrix_i]],
                     clustering_distance_rows = "euclidean",
                     cluster_rows = TRUE,
                     # cluster_cols = FALSE,
                     breaks = seq(-rg, rg, length.out = 100),
                     clustering_distance_cols = "euclidean",
                     clustering_method = "complete",
                     # labels_row = row_labels,
                     # labels_col = col_labels,
                     color = my_colors,
                     show_rownames = FALSE,
                     show_colnames = FALSE,
                     annotation_col = metadata_col[, vtr_annotations[[heatmap_j]], drop = FALSE],
                     annotation_row = metadata_col[, vtr_annotations[[heatmap_j]], drop = FALSE],
                     annotation_colors = ann_colors,
                     border_color  = 'black',
                     main = heatmap_title,
                     fontsize = 8,
                     filename = paste0(heatmap_file, "_", heatmap_j, ".pdf"),
                     width = 12,
                     height = 10)
        }

        reordered_matrix <- result_matrices[[heatmap_i]][[matrix_i]][out$tree_row[["order"]], out$tree_col[["order"]]]
        
        write_xlsx(data.frame(VTR = rownames(reordered_matrix), reordered_matrix, check.names = FALSE), paste0(heatmap_file, ".xlsx"))
    }
}

### Fig. 2A

In [ ]:
library(dplyr)
library(tibble)
library(readxl)
library(pheatmap)
library(ggplot2)
library(RColorBrewer)
library(msigdbr)
library(fgsea)
library(enrichplot)
library(DOSE)
library(clusterProfiler)
library(stringr)
library(vegan)
library(openxlsx)
library(writexl)

In [ ]:
msigdb <- msigdbr(species = 'Homo sapiens', category = 'C2')
msigdb <- msigdb %>% filter(gs_subcat %in% c('CP:REACTOME'))
unique(msigdb$gs_subcat)

In [ ]:
# Only use strategy2 because it's the best option after checking summary and cooncordancy_summary tables
vtr_df <- read_excel('/rprojectnb/cancergrp/brb/raw_data/VTR_INFORMATION.xlsx')

#folder_path <- paste0("/rprojectnb/cancergrp/brb/reproducibility/intermediate_files/GSEA_objects/")
folder_path <- paste0("/rprojectnb/cancergrp/brb/intermediate_files/new_filtered_gsea_runs/gsea_tables/")

table_file <- paste0(folder_path, "master_table.xlsx")
pvalue_file <- paste0(folder_path, "pvalue_table.xlsx")

#### For unstimulated

In [ ]:
condition_id <- 'unstimulated'
score_matrix_saved <- read.xlsx(table_file)
p_value_matrix <- read.xlsx(pvalue_file)

In [ ]:
dim(score_matrix_saved)
dim(p_value_matrix)

In [ ]:
score_matrix_saved <- score_matrix_saved %>% select(-ctrl_unstimulated_Batch3.Batch4.Batch5.Batch6)
p_value_matrix <- p_value_matrix %>% select(-ctrl_unstimulated_Batch3.Batch4.Batch5.Batch6)


In [ ]:
dim(score_matrix_saved)
dim(p_value_matrix)

In [ ]:

# Exclude the "Paths" column from the numeric operations and retain it separately
score_paths <- score_matrix_saved[,"Paths"]
numeric_score_matrix <- score_matrix_saved[,-which(colnames(score_matrix_saved) == "Paths")]
numeric_pvalue_matrix <- p_value_matrix[,-which(colnames(p_value_matrix) == "Paths")]

# Set NES values to 0 where p-value is greater than 0.1
numeric_score_matrix[numeric_pvalue_matrix > 0.05] <- 0

# Reattach the "Paths" column to the modified numeric matrix
score_matrix_saved <- cbind(Paths = score_paths, numeric_score_matrix)



In [ ]:
dim(score_matrix_saved)                                                           

In [ ]:
# Filter columns by condition (Just unstimulated in this case)
score_matrix_saved <- score_matrix_saved[, c("Paths", grep(paste0("_", condition_id), 
                                                           colnames(score_matrix_saved), 
                                                           value = TRUE))]


In [ ]:
dim(score_matrix_saved)

In [ ]:
# From DF to Matrix
score_matrix2 = as.matrix(score_matrix_saved)

# GSEA Paths as rownames
rownames(score_matrix2) = score_matrix2[,1]

# Remove GSEA Path which was first column
score_matrix2 = score_matrix2[, -1]

# Replace NA values with 0
score_matrix2[is.na(score_matrix2)] = 0

# Filter rows with non-zeros in 3 or more columns (samples)
score_matrix2 <- score_matrix2[rowSums(score_matrix2 != 0) >= 3, ]

# Get variable for GSEA path names
rownames_score <- rownames(score_matrix2)

# Convert to numeric all values
score_matrix2 <- apply(score_matrix2, 2, as.numeric)

# Add GSEA Paths to rownames
rownames(score_matrix2) = rownames_score

In [ ]:
# Create a dataframe where each column is converted to a row 
metadata_col <- data.frame(id = colnames(score_matrix2))

# Add Batch id, condition, and vtr
metadata_col$batch_id = str_split_fixed(metadata_col$id, '\\_', 3)[,3]
metadata_col$condition = str_split_fixed(metadata_col$id, '\\_', 3)[,2]
metadata_col$vtr = str_split_fixed(metadata_col$id, '\\_', 3)[,1]

# Add Ctrl or Viral family
metadata_col$virus_type <- ifelse(
  metadata_col$vtr == "ctrl", 
  "ctrl", 
  vtr_df$`Viral family`[match(metadata_col$vtr, vtr_df$code)]
)

# Copy ID to rownames
rownames(metadata_col) = metadata_col$id

In [ ]:
# Crete palette colors
n_virus_type = length(unique(metadata_col$virus_type))
n_batch_id = length(unique(metadata_col$batch_id))
n_condition = length(unique(metadata_col$condition))

ann_colors <- list(
    virus_type = setNames(colorRampPalette(brewer.pal(n_virus_type, "Set3"))(n_virus_type), unique(metadata_col$virus_type)),
    batch_id = setNames(colorRampPalette(brewer.pal(n_batch_id, "Dark2"))(n_batch_id), unique(metadata_col$batch_id)),
    condition = setNames(colorRampPalette(brewer.pal(n_condition, "Paired"))(n_condition), unique(metadata_col$condition))
)

# Calculate dynamic min and max values based on the matrix
min_value <- min(score_matrix2, na.rm = TRUE)
max_value <- max(score_matrix2, na.rm = TRUE)

# Create a color palette with 100 colors, centering white at 0
my_colors <- colorRampPalette(c("navy", "white", "firebrick3"))(100)

In [ ]:
folder_path2 = './'
heatmap_file <- paste0(folder_path2, "Ctrl_removed_2025_heatmap_",
                       condition_id)
matrix_file <- paste0(folder_path2, "Ctrl_removed_2025_matrix_",
                       condition_id)

#### CSI calculation

In [ ]:
library(parallel)
library(doParallel)
library(foreach)
calculate_csi_matrix <- function(dist_obj, num_cores = NULL) {
  # Get sample names
  sample_names <- labels(dist_obj)
  n <- length(sample_names)
  
  # Determine number of cores to use
  if (is.null(num_cores)) {
    num_cores <- max(1, detectCores() - 1)  # Use all cores minus one
  }
  
  # Convert distance object to matrix (necessary for efficient access)
  dist_matrix <- as.matrix(dist_obj)
  
  # Create empty matrix to store CSI results
  csi_matrix <- matrix(0, nrow = n, ncol = n)
  dimnames(csi_matrix) <- list(sample_names, sample_names)
  
  # Setup cluster for parallelization
  cl <- makeCluster(num_cores)
  registerDoParallel(cl)
  
  # Create indices for all pairs to calculate (only half of the matrix)
  pairs <- expand.grid(i = 1:n, j = 1:n)
  pairs <- pairs[pairs$i < pairs$j, ]  # We only need the triangular half
  
  # Calculate CSI in parallel
  results <- foreach(idx = 1:nrow(pairs), .combine = "c") %dopar% {
    i <- pairs$i[idx]
    j <- pairs$j[idx]
    
    sample_a <- sample_names[i]
    sample_b <- sample_names[j]
    
    # Distance between A and B
    dist_ab <- dist_matrix[sample_a, sample_b]
    
    # Counter for CSI
    csi_count <- 0
    
    # For each other sample, calculate the maximum of distances
    for (k in 1:n) {
      sample_x <- sample_names[k]
      # Skip samples A and B
      if (sample_x != sample_a && sample_x != sample_b) {
        # Distances
        dist_ax <- dist_matrix[sample_a, sample_x]
        dist_bx <- dist_matrix[sample_b, sample_x]
        
        # Calculate the maximum
        max_dist <- max(dist_ax, dist_bx)
        
        # Increment counter if this maximum is less than dist_ab
        if (max_dist < dist_ab) {
          csi_count <- csi_count + 1
        }
      }
    }
    
    csi_count
  }
  
  # Stop the cluster
  stopCluster(cl)
  
  # Fill the CSI matrix with results
  for (idx in 1:nrow(pairs)) {
    i <- pairs$i[idx]
    j <- pairs$j[idx]
    csi_matrix[i, j] <- results[idx]
    csi_matrix[j, i] <- results[idx]  # The CSI matrix is symmetric
  }
  
  return(csi_matrix)
}

#### Correlation distance for samples 

In [ ]:
cor_samples <- as.dist(1 - cor(score_matrix2, method = 'pearson'))

#### Calculate CSI just for samples

In [ ]:
## Calculate CSI
csi_samples <- calculate_csi_matrix(as.dist(cor(score_matrix2, method = 'pearson')))

## Normalize over n-2
csi_samples <- csi_samples/(ncol(score_matrix2)-2)

## From Matrix to Distance object
csi_samples_dist <- as.dist(csi_samples)

#### Invert CSI to have low values == samples more similar and should cluster together

In [ ]:
inverted_csi_samples <- 1 - csi_samples

inverted_csi_samples_dist <- as.dist(inverted_csi_samples)


In [ ]:
csi_samples

In [ ]:
write.xlsx( x = inverted_csi_samples,
           file = 'Unstimulated_samples_matrix_CSI_from_correlation_paths.xlsx',
           rowNames  = TRUE,
           colNames  = TRUE,
           keepNA    = FALSE
          )

In [ ]:
pheatmap(csi_samples,
         cluster_rows = TRUE,
         cluter_cols = TRUE,
         breaks = seq(0, 1, length.out = 101),
         clustering_distance_cols = inverted_csi_samples_dist,
         clustering_distance_rows = inverted_csi_samples_dist,
         clustering_method = "complete",
         color = my_colors,
         show_rownames = TRUE,
         show_colnames = TRUE,
         annotation_col = metadata_col[, c('virus_type', 'batch_id'), drop = FALSE],
         annotation_row = metadata_col[, c('virus_type', 'batch_id'), drop = FALSE],
         annotation_colors = ann_colors,
         border_color  = 'black',
         main = "GSEA NES Heatmap",
         fontsize = 8,
         filename = 'Unstimulated_samples_matrix_CSI_from_correlation_paths.pdf',
         width = 12,
         height = 10)

#### Euclidean distance for paths

In [ ]:
dist_paths <- dist(score_matrix2, method = "euclidean")

#### Create pheatmap with csi distances (inverted_csi_paths_dist, inverted_csi_samples_dist)

In [ ]:
rg <- max(abs(score_matrix2))
## Plot heatmap
out <- pheatmap(score_matrix2,
         clustering_distance_rows = dist_paths,
         cluster_rows = TRUE,
         breaks = seq(-rg, rg, length.out = 101),
         clustering_distance_cols = inverted_csi_samples_dist,
         clustering_method = "complete",
         color = my_colors,
         show_rownames = FALSE,
         show_colnames = FALSE,
         annotation_col = metadata_col[, c('virus_type', 'batch_id'), drop = FALSE],
         annotation_colors = ann_colors,
         border_color  = 'black',
         main = "GSEA NES Heatmap",
         fontsize = 8,
         filename = paste0(heatmap_file, '_samples_CSI_from_correlation_paths_euclidean_method_complete', ".pdf"),
         width = 12,
         height = 10)
reordered_matrix <- score_matrix2[out$tree_row[["order"]], out$tree_col[["order"]]]
write.xlsx(
  x         = reordered_matrix,
  file      = paste0(matrix_file, '_samples_CSI_from_correlation_paths_euclidean_method_complete', '.xlsx'),
  rowNames  = TRUE,
  colNames  = TRUE,
  keepNA    = FALSE
)


In [ ]:
rg <- max(abs(score_matrix2))
## Plot heatmap
out <- pheatmap(score_matrix2,
         clustering_distance_rows = dist_paths,
         cluster_rows = TRUE,
         breaks = seq(-rg, rg, length.out = 101),
         clustering_distance_cols = as.dist(1 - cor(score_matrix2, method = 'pearson')),
         clustering_method = "complete",
         color = my_colors,
         show_rownames = FALSE,
         show_colnames = FALSE,
         annotation_col = metadata_col[, c('virus_type', 'batch_id'), drop = FALSE],
         annotation_colors = ann_colors,
         border_color  = 'black',
         main = "GSEA NES Heatmap",
         fontsize = 8,
         filename = paste0(heatmap_file, '_samples_correlation_paths_euclidean_method_complete', ".pdf"),
         width = 12,
         height = 10)
reordered_matrix <- score_matrix2[out$tree_row[["order"]], out$tree_col[["order"]]]
write.xlsx(
  x         = reordered_matrix,
  file      = paste0(matrix_file, '_samples_correlation_paths_euclidean_method_complete', '.xlsx'),
  rowNames  = TRUE,
  colNames  = TRUE,
  keepNA    = FALSE
)

### Stim_and_Unstim_samples_matrix_CSI_from_correlation_paths.xlsx

In [ ]:
library(dplyr)
library(tibble)
library(readxl)
library(pheatmap)
library(ggplot2)
library(RColorBrewer)
library(msigdbr)
library(fgsea)
library(enrichplot)
library(DOSE)
library(clusterProfiler)
library(stringr)
library(vegan)
library(openxlsx)
library(writexl)

In [ ]:
msigdb <- msigdbr(species = 'Homo sapiens', category = 'C2')
msigdb <- msigdb %>% filter(gs_subcat %in% c('CP:REACTOME'))
unique(msigdb$gs_subcat)

In [ ]:
# Only use strategy2 because it's the best option after checking summary and cooncordancy_summary tables
vtr_df <- read_excel('/rprojectnb/cancergrp/brb/raw_data/VTR_INFORMATION.xlsx')

folder_path <- paste0("/rprojectnb/cancergrp/brb/intermediate_files/new_filtered_gsea_runs/gsea_tables/")

table_file <- paste0(folder_path, "master_table.xlsx")
pvalue_file <- paste0(folder_path, "pvalue_table.xlsx")

#### For unstimulated

In [ ]:
condition_id <- 'unstimulated'
score_matrix_saved <- read.xlsx(table_file)
p_value_matrix <- read.xlsx(pvalue_file)

In [ ]:
dim(score_matrix_saved)
dim(p_value_matrix)

In [ ]:
score_matrix_saved <- score_matrix_saved %>% select(-ctrl_unstimulated_Batch3.Batch4.Batch5.Batch6)
p_value_matrix <- p_value_matrix %>% select(-ctrl_unstimulated_Batch3.Batch4.Batch5.Batch6)


In [ ]:
dim(score_matrix_saved)
dim(p_value_matrix)

In [ ]:

# Exclude the "Paths" column from the numeric operations and retain it separately
score_paths <- score_matrix_saved[,"Paths"]
numeric_score_matrix <- score_matrix_saved[,-which(colnames(score_matrix_saved) == "Paths")]
numeric_pvalue_matrix <- p_value_matrix[,-which(colnames(p_value_matrix) == "Paths")]

# Set NES values to 0 where p-value is greater than 0.1
numeric_score_matrix[numeric_pvalue_matrix > 0.05] <- 0

# Reattach the "Paths" column to the modified numeric matrix
score_matrix_saved <- cbind(Paths = score_paths, numeric_score_matrix)



In [ ]:
dim(score_matrix_saved)

In [ ]:
# From DF to Matrix
score_matrix2 = as.matrix(score_matrix_saved)

# GSEA Paths as rownames
rownames(score_matrix2) = score_matrix2[,1]

# Remove GSEA Path which was first column
score_matrix2 = score_matrix2[, -1]

# Replace NA values with 0
score_matrix2[is.na(score_matrix2)] = 0

# Filter rows with non-zeros in 3 or more columns (samples)
score_matrix2 <- score_matrix2[rowSums(score_matrix2 != 0) >= 3, ]

# Get variable for GSEA path names
rownames_score <- rownames(score_matrix2)

# Convert to numeric all values
score_matrix2 <- apply(score_matrix2, 2, as.numeric)

## Add GSEA Paths to rownames
rownames(score_matrix2) = rownames_score

In [ ]:
# Create a dataframe where each column is converted to a row 
metadata_col <- data.frame(id = colnames(score_matrix2))

# Add Batch id, condition, and vtr
metadata_col$batch_id = str_split_fixed(metadata_col$id, '\\_', 3)[,3]
metadata_col$condition = str_split_fixed(metadata_col$id, '\\_', 3)[,2]
metadata_col$vtr = str_split_fixed(metadata_col$id, '\\_', 3)[,1]

# Add Ctrl or Viral family
metadata_col$virus_type <- ifelse(
  metadata_col$vtr == "ctrl", 
  "ctrl", 
  vtr_df$`Viral family`[match(metadata_col$vtr, vtr_df$code)]
)

# Copy ID to rownames
rownames(metadata_col) = metadata_col$id

In [ ]:
# Crete palette colors
n_virus_type = length(unique(metadata_col$virus_type))
n_batch_id = length(unique(metadata_col$batch_id))
n_condition = length(unique(metadata_col$condition))

ann_colors <- list(
    virus_type = setNames(colorRampPalette(brewer.pal(n_virus_type, "Set3"))(n_virus_type), unique(metadata_col$virus_type)),
    batch_id = setNames(colorRampPalette(brewer.pal(n_batch_id, "Dark2"))(n_batch_id), unique(metadata_col$batch_id)),
    condition = setNames(colorRampPalette(brewer.pal(n_condition, "Paired"))(n_condition), unique(metadata_col$condition))
)

# Calculate dynamic min and max values based on the matrix
min_value <- min(score_matrix2, na.rm = TRUE)
max_value <- max(score_matrix2, na.rm = TRUE)

# Create a color palette with 100 colors, centering white at 0
my_colors <- colorRampPalette(c("navy", "white", "firebrick3"))(100)

In [ ]:
folder_path2 = './'
heatmap_file <- paste0(folder_path2, "Ctrl_removed_2025_heatmap_",
                       condition_id)
matrix_file <- paste0(folder_path2, "Ctrl_removed_2025_matrix_",
                       condition_id)

#### CSI calculation

In [ ]:
library(parallel)
library(doParallel)
library(foreach)
calculate_csi_matrix <- function(dist_obj, num_cores = NULL) {
  # Get sample names
  sample_names <- labels(dist_obj)
  n <- length(sample_names)
  
  # Determine number of cores to use
  if (is.null(num_cores)) {
    num_cores <- max(1, detectCores() - 1)  # Use all cores minus one
  }
  
  # Convert distance object to matrix (necessary for efficient access)
  dist_matrix <- as.matrix(dist_obj)
  
  # Create empty matrix to store CSI results
  csi_matrix <- matrix(0, nrow = n, ncol = n)
  dimnames(csi_matrix) <- list(sample_names, sample_names)
  
  # Setup cluster for parallelization
  cl <- makeCluster(num_cores)
  registerDoParallel(cl)
  
  # Create indices for all pairs to calculate (only half of the matrix)
  pairs <- expand.grid(i = 1:n, j = 1:n)
  pairs <- pairs[pairs$i < pairs$j, ]  # We only need the triangular half
  
  # Calculate CSI in parallel
  results <- foreach(idx = 1:nrow(pairs), .combine = "c") %dopar% {
    i <- pairs$i[idx]
    j <- pairs$j[idx]
    
    sample_a <- sample_names[i]
    sample_b <- sample_names[j]
    
    # Distance between A and B
    dist_ab <- dist_matrix[sample_a, sample_b]
    
    # Counter for CSI
    csi_count <- 0
    
    # For each other sample, calculate the maximum of distances
    for (k in 1:n) {
      sample_x <- sample_names[k]
      # Skip samples A and B
      if (sample_x != sample_a && sample_x != sample_b) {
        # Distances
        dist_ax <- dist_matrix[sample_a, sample_x]
        dist_bx <- dist_matrix[sample_b, sample_x]
        
        # Calculate the maximum
        max_dist <- max(dist_ax, dist_bx)
        
        # Increment counter if this maximum is less than dist_ab
        if (max_dist < dist_ab) {
          csi_count <- csi_count + 1
        }
      }
    }
    
    csi_count
  }
  
  # Stop the cluster
  stopCluster(cl)
  
  # Fill the CSI matrix with results
  for (idx in 1:nrow(pairs)) {
    i <- pairs$i[idx]
    j <- pairs$j[idx]
    csi_matrix[i, j] <- results[idx]
    csi_matrix[j, i] <- results[idx]  # The CSI matrix is symmetric
  }
  
  return(csi_matrix)
}

#### Correlation distance for samples 

In [ ]:
cor_samples <- as.dist(1 - cor(score_matrix2, method = 'pearson'))

#### Calculate CSI just for samples

In [ ]:
## Calculate CSI
csi_samples <- calculate_csi_matrix(as.dist(cor(score_matrix2, method = 'pearson')))

## Normalize over n-2
csi_samples <- csi_samples/(ncol(score_matrix2)-2)

## From Matrix to Distance object
csi_samples_dist <- as.dist(csi_samples)

#### Invert CSI to have low values == samples more similar and should cluster together

In [ ]:
inverted_csi_samples <- 1 - csi_samples

inverted_csi_samples_dist <- as.dist(inverted_csi_samples)


In [ ]:
csi_samples

In [ ]:
write.xlsx( x = inverted_csi_samples,
           file = 'Unstimulated_samples_matrix_CSI_from_correlation_paths.xlsx',
           rowNames  = TRUE,
           colNames  = TRUE,
           keepNA    = FALSE
          )